# VGG16 fine-tuning for defect detection

This section builds a VGG16 model with ImageNet weights, lets you choose any valid input shape (HxWx3), and fine-tune only the last layers you specify.

Two variants are trained, one per input resolution:

- **HR** classifies high-resolution and super-resolved images.
- **LR** classifies low-resolution images at their native size, and provides the baseline of the defect detection pipeline.

A single classifier cannot serve both roles. It extracts `VGG_PATCH_SIZE` patches from whatever image it receives, so feeding it a 239x239 LR image after training on 478x478 frames halves the apparent object scale and shrinks the number of voting patches. That scale mismatch penalises the baseline for reasons unrelated to resolution. Training one classifier per resolution removes it.

Both variants share every hyperparameter and every image-level partition, so the only difference between them is the resolution of their input.

In [ ]:
import os
import sys
import datetime

import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))
from srlib.defect_detection.vgg16 import FineTunedVGG16
from srlib.dataset.loading import load_vgg16_dataset
from srlib.dataset.visualization import plot_classification_patches
from srlib.deep_learning.visualization import plot_vgg16_training_curves
from srlib.constants import (
    CLASS_LABELS_PATH,
    DL_RESULTS_DIR,
    HR_ROOT,
    LR_ROOT,
    TIMESTAMP_FORMAT,
    VGG16_FIT_PARAMS,
    VGG16_SETUP_PARAMS,
    VGG16_SOURCES,
    VGG_PATCH_SIZE,
    VGG_STRIDE,
)

In [ ]:
# X -> image patches (model input)
# y -> class labels (target)
# Both variants derive from the same image-level partition, so the HR and LR
# test sets contain the very same images at two resolutions.
datasets = {}

for source in VGG16_SOURCES:
    datasets[source] = load_vgg16_dataset(
        HR_ROOT,
        LR_ROOT,
        CLASS_LABELS_PATH,
        source=source,
        patch_size=VGG_PATCH_SIZE,
        stride=VGG_STRIDE,
    )

In [ ]:
# Patches have the same pixel size in both rows, so the visible difference
# is the object scale each classifier learns to recognise.
_ = plot_classification_patches(
    {s: datasets[s][0] for s in VGG16_SOURCES},
    {s: datasets[s][1] for s in VGG16_SOURCES},
    count=4,
    seed=0,
)

X shape: (14022, 96, 96, 3), Y shape: (14022,)
X_train shape: (10095, 96, 96, 3), y_train shape: (10095,)
X_val shape: (1122, 96, 96, 3), y_val shape: (1122,)
X_test shape: (2805, 96, 96, 3), y_test shape: (2805,)
Class distribution: {0: 5446, 1: 8576}


In [ ]:
models_by_source = {}

for source in VGG16_SOURCES:
    X_train, y_train = datasets[source][0], datasets[source][1]

    print(f"Building VGG16 for {source.upper()} patches")
    model = FineTunedVGG16()
    model.setup_model(
        input_shape=X_train.shape[1:],
        num_classes=np.unique(y_train).shape[0],
        **VGG16_SETUP_PARAMS,
    )

    models_by_source[source] = model

Model: "vgg16_finetune"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 vgg16 (Functional)          (None, 3, 3, 512)         14714688  
                                                                 
 gap (GlobalAveragePooling2D  (None, 512)              0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense (Dense)               (None, 256)               131328    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                    

In [ ]:
# Each entry holds the (head, fine-tuning) pair of histories, since the two
# phases are separate training runs with different learning rates.
histories_by_source = {}

for source in VGG16_SOURCES:
    X_train, y_train, X_val, y_val = datasets[source][:4]

    print(f"Training VGG16 on {source.upper()} patches")
    histories_by_source[source] = models_by_source[source].fit_two_phases(
        X_train, y_train,
        X_val, y_val,
        **VGG16_FIT_PARAMS,
    )

Epoch 1/150
316/316 [==============================] - 18s 41ms/step - loss: 0.3994 - accuracy: 0.8268 - val_loss: 0.2869 - val_accuracy: 0.8806 - lr: 0.0010
Epoch 2/150
316/316 [==============================] - 11s 36ms/step - loss: 0.3491 - accuracy: 0.8527 - val_loss: 0.2679 - val_accuracy: 0.8913 - lr: 0.0010
Epoch 3/150
316/316 [==============================] - 11s 35ms/step - loss: 0.3295 - accuracy: 0.8635 - val_loss: 0.2876 - val_accuracy: 0.8779 - lr: 0.0010
Epoch 4/150
316/316 [==============================] - 11s 36ms/step - loss: 0.3194 - accuracy: 0.8690 - val_loss: 0.2568 - val_accuracy: 0.8886 - lr: 0.0010
Epoch 5/150
316/316 [==============================] - 12s 37ms/step - loss: 0.3096 - accuracy: 0.8726 - val_loss: 0.2529 - val_accuracy: 0.8939 - lr: 0.0010
Epoch 6/150
316/316 [==============================] - 12s 37ms/step - loss: 0.3092 - accuracy: 0.8724 - val_loss: 0.2726 - val_accuracy: 0.8850 - lr: 0.0010
Epoch 7/150
316/316 [==============================]

In [ ]:
# One timestamp for the whole run: both variants are told apart by their
# source tag, so model_registry resolves them as a single VGG16 run.
timestamp = datetime.datetime.now().strftime(TIMESTAMP_FORMAT)
metrics_by_source = {}

for source in VGG16_SOURCES:
    X_test, y_test = datasets[source][4], datasets[source][5]
    head_history, finetune_history = histories_by_source[source]

    _, _, metrics_by_source[source] = models_by_source[source].evaluate_and_save(
        X_test, y_test, head_history, finetune_history, source,
        timestamp=timestamp,
    )

88/88 [==============================] - 2s 22ms/step - loss: 0.2029 - accuracy: 0.9205
Loss: 0.2029, Accuracy: 0.9205
88/88 [==============================] - 2s 20ms/step - loss: 0.2029 - accuracy: 0.9205
Loss: 0.2029, Accuracy: 0.9205


## Training curves

Both variants on the same axes, so the effect of the input resolution is visible directly. Training is solid, validation dashed, the dotted horizontal line is the test score and the dash-dotted vertical line marks the epoch the backbone was unfrozen, where a step in the curve is expected.

In [ ]:
plot_vgg16_training_curves(metrics_by_source, save_path=DL_RESULTS_DIR)